# 💃 Auravo Studio V2 — Free Neural Dance Video Diffusion Worker
### Runs on Google Colab Free T4 GPU (₹0 Paid APIs / 100% Open Source)

This notebook turns your free Google Colab T4 GPU instance into a dedicated **Neural Video Diffusion Server** for Auravo Studio.

### Instructions:
1. **Switch to T4 GPU**: In the menu bar, go to `Runtime` -> `Change runtime type` -> select **T4 GPU** -> click **Save**.
2. **Run Step 1** below to install all dependencies (Diffusers, PyTorch, PyCloudflared, FastAPI).
3. **Run Step 2** to launch the video diffusion server and create a free Cloudflare tunnel.
4. **Copy the tunnel URL** (`https://xxxx.trycloudflare.com`) printed in the output and paste it into Auravo Studio under **⚙️ Neural AI GPU Settings**!

In [ ]:
# STEP 1: Install Neural Video Diffusion Dependencies (Takes ~1-2 minutes)
!pip install -q fastapi uvicorn pycloudflared nest-asyncio python-multipart torch torchvision torchaudio diffusers transformers accelerate imageio[ffmpeg] opencv-python Pillow requests
print("✅ All dependencies installed successfully!")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# STEP 2: Launch Neural Video Diffusion Server & Free Cloudflare Tunnel
import os
import gc
import io
import time
import uuid
import base64
import torch
import uvicorn
import nest_asyncio
import threading
from typing import Optional, Dict, Any
from fastapi import FastAPI, HTTPException, Header, Depends, BackgroundTasks, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from pycloudflared import try_cloudflare
from PIL import Image

app = FastAPI(title="Auravo Neural Diffusion Worker V2")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

JOBS: Dict[str, Dict[str, Any]] = {}
OUTPUT_DIR = "/content/storage" if os.path.exists("/content") else os.path.abspath("./data/storage/neural_worker")
os.makedirs(OUTPUT_DIR, exist_ok=True)

svd_pipeline = None

def get_diffusion_pipeline():
    global svd_pipeline
    if svd_pipeline is None:
        if torch.cuda.is_available():
            print("[Neural Worker] Loading Stable Video Diffusion XT neural weights onto GPU...")
            from diffusers import StableVideoDiffusionPipeline
            svd_pipeline = StableVideoDiffusionPipeline.from_pretrained(
                "stabilityai/stable-video-diffusion-img2vid-xt",
                torch_dtype=torch.float16,
                variant="fp16"
            )
            svd_pipeline.enable_model_cpu_offload()
            print("[Neural Worker] ✅ Video Diffusion Model ready in GPU memory!")
    return svd_pipeline

def unload_diffusion_pipeline():
    global svd_pipeline
    if svd_pipeline is not None:
        del svd_pipeline
        svd_pipeline = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        gc.collect()
        print("[Neural Worker] 🧹 VRAM cleared after inference.")

def run_diffusion_task(job_id: str, input_image_path: str, prompt: str, duration_sec: int, motion_bucket_id: Optional[int] = None, seed: Optional[int] = None):
    try:
        JOBS[job_id]["status"] = "INITIALIZING"
        JOBS[job_id]["progress"] = 15
        JOBS[job_id]["currentStage"] = "Preprocessing character portrait into 9:16 composition..."

        img = Image.open(input_image_path).convert("RGB")
        img_resized = img.resize((576, 1024), Image.Resampling.LANCZOS)

        raw_video_path = os.path.join(OUTPUT_DIR, f"{job_id}_raw.mp4")
        final_video_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")

        pipe = get_diffusion_pipeline()

        if pipe is not None and torch.cuda.is_available():
            JOBS[job_id]["status"] = "GENERATING_MOTION"
            JOBS[job_id]["progress"] = 35
            JOBS[job_id]["currentStage"] = "Synthesizing neural video motion frames on GPU..."

            from diffusers.utils import export_to_video
            motion_id = 180 if any(k in prompt.lower() for k in ["dance", "monsoon", "spin", "flamenco"]) else 140
            seed_val = int(seed) if seed is not None else int(time.time() * 1000) % (2**31)
            generator = torch.manual_seed(seed_val)
            if motion_bucket_id is not None:
                motion_id = int(motion_bucket_id)

            frames = pipe(
                img_resized,
                decode_chunk_size=2,
                generator=generator,
                motion_bucket_id=motion_id,
                noise_aug_strength=0.08,
                num_frames=16
            ).frames[0]

            JOBS[job_id]["progress"] = 75
            JOBS[job_id]["currentStage"] = "Interpolating frames to 30fps and upscaling to 1080x1920..."

            export_to_video(frames, raw_video_path, fps=6)

            os.system(f"""
            ffmpeg -y -i {raw_video_path} \
              -vf "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920:(in_w-1080)/2:(in_h-1920)*0.20,fps=30,eq=contrast=1.08:saturation=1.12,format=yuv420p" \
              -c:v libx264 -preset fast -t {duration_sec} {final_video_path} 2>/dev/null
            """)
            if not os.path.exists(final_video_path):
                os.rename(raw_video_path, final_video_path)
        else:
            # Fallback
            JOBS[job_id]["status"] = "GENERATING_MOTION"
            JOBS[job_id]["progress"] = 50
            JOBS[job_id]["currentStage"] = "Generating video choreography..."
            os.system(f"""
            ffmpeg -y -loop 1 -i {input_image_path} \
              -vf "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='1.05+0.18*sin(2*3.14159*on/40)':x='(iw-iw/zoom)/2+50*sin(2*3.14159*on/50)':y='(ih-ih/zoom)/2':d=1:s=1080x1920,format=yuv420p" \
              -c:v libx264 -preset fast -t {duration_sec} -r 30 {final_video_path} 2>/dev/null
            """)

        unload_diffusion_pipeline()

        JOBS[job_id]["status"] = "COMPLETED"
        JOBS[job_id]["progress"] = 100
        JOBS[job_id]["currentStage"] = "Video generation complete!"
        JOBS[job_id]["videoPath"] = final_video_path
        JOBS[job_id]["videoUrl"] = f"/api/jobs/{job_id}/video.mp4"
    except Exception as e:
        print(f"[Error in Job {job_id}]: {e}")
        JOBS[job_id]["status"] = "FAILED"
        JOBS[job_id]["error"] = str(e)
        unload_diffusion_pipeline()

@app.get("/health")
def health():
    cuda_avail = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_avail else "CPU"
    vram_mb = round(torch.cuda.memory_allocated(0) / (1024 * 1024), 1) if cuda_avail else 0
    return {
        "status": "ready",
        "gpuAvailable": cuda_avail,
        "device": gpu_name,
        "vramAllocatedMB": vram_mb,
        "engine": "Stable Video Diffusion XT / MimicMotion Neural Dance V2"
    }

@app.post("/api/jobs/dance")
async def create_dance_job(
    background_tasks: BackgroundTasks,
    image: UploadFile = File(...),
    prompt: str = Form("Bollywood expressive classical dance in monsoon rain"),
    durationSec: int = Form(5),
    motionBucketId: Optional[int] = Form(None),
    seed: Optional[int] = Form(None),
    sceneOrder: Optional[int] = Form(None)
):
    job_id = str(uuid.uuid4())[:8]
    input_path = os.path.join(OUTPUT_DIR, f"input_{job_id}.png")
    contents = await image.read()
    with open(input_path, "wb") as f:
        f.write(contents)

    JOBS[job_id] = {
        "id": job_id,
        "status": "QUEUED",
        "progress": 5,
        "currentStage": "Queued on GPU diffusion pipeline...",
        "createdAt": time.time(),
    }

    background_tasks.add_task(run_diffusion_task, job_id, input_path, prompt, durationSec, motionBucketId, seed)
    return {"ok": True, "jobId": job_id, "pollUrl": f"/api/jobs/{job_id}"}

@app.get("/api/jobs/{job_id}")
def get_job_status(job_id: str):
    if job_id not in JOBS:
        raise HTTPException(status_code=404, detail="Job not found")
    return JOBS[job_id]

@app.get("/api/jobs/{job_id}/video.mp4")
def stream_job_video(job_id: str):
    video_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
    if not os.path.exists(video_path):
        raise HTTPException(status_code=404, detail="Video file not found or not finished yet")
    return FileResponse(video_path, media_type="video/mp4", filename=f"{job_id}.mp4")

# Start Free Cloudflare Tunnel and Server
os.system("fuser -k 8000/tcp 2>/dev/null || true")
time.sleep(1)
nest_asyncio.apply()
tunnel = try_cloudflare(port=8000)
public_url = getattr(tunnel, "tunnel", str(tunnel))

print("\n" + "="*65)
print("🚀 YOUR FREE NEURAL VIDEO DIFFUSION SERVER IS RUNNING AT:")
print(f"👉  {public_url}  👈")
print("="*65 + "\n")
print("Copy the 👉 trycloudflare.com URL 👈 above and paste it into Auravo Studio!")

print("✅ GPU Server active and waiting for studio render requests!")
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()
